In [7]:
import os
import json
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from PIL import Image

###########################################
# 1. Dataset e Collate Function
###########################################
class MultiTaskObjectDetectionDataset(Dataset):
    def __init__(self, images_dir, labels_dir, transform=None):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.transform = transform

        # Filtra as imagens que possuem JSON correspondente
        all_ids = [f.split('.')[0] for f in os.listdir(images_dir) if f.endswith('.jpg')]
        self.image_ids = []
        missing = []
        for img_id in all_ids:
            json_path = os.path.join(labels_dir, img_id + '.json')
            if os.path.exists(json_path):
                self.image_ids.append(img_id)
            else:
                missing.append(img_id)
        if missing:
            print("Os seguintes arquivos de imagem não possuem JSON:")
            for m in missing:
                print(m)
        else:
            print("Todas as imagens possuem JSON correspondente.")

        # Carrega os mapeamentos (assumindo que estão na pasta 'helpers')
        with open('helpers/categories.json', 'r') as f:
            self.category_to_label = json.load(f)
        with open('helpers/weather.json', 'r') as f:
            self.weather_to_label = json.load(f)
        with open('helpers/scene.json', 'r') as f:
            self.scene_to_label = json.load(f)
        with open('helpers/timeofday.json', 'r') as f:
            self.timeofday_to_label = json.load(f)

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        try:
            image = Image.open(os.path.join(self.images_dir, image_id + ".jpg")).convert("RGB")
            with open(os.path.join(self.labels_dir, image_id + ".json")) as f:
                data = json.load(f)
        except Exception as e:
            print(f"❌ Erro ao carregar {image_id}: {e}")
            return None

        boxes, labels = [], []
        for obj in data["frames"][0]["objects"]:
            category = obj.get("category")
            if category not in self.category_to_label:
                continue
            bbox = None
            if "box2d" in obj:
                b = obj["box2d"]
                bbox = [b["x1"], b["y1"], b["x2"], b["y2"]]
            elif "poly2d" in obj:
                pts_raw = obj["poly2d"]
                if isinstance(pts_raw[0], dict) and "vertices" in pts_raw[0]:
                    pts = pts_raw[0]["vertices"]
                else:
                    pts = [(p[0], p[1]) for p in pts_raw if isinstance(p, (list, tuple)) and len(p) >= 2]
                if len(pts) >= 2:
                    xs, ys = zip(*pts)
                    bbox = [min(xs), min(ys), max(xs), max(ys)]
            if bbox and bbox[2] > bbox[0] and bbox[3] > bbox[1]:
                boxes.append(bbox)
                labels.append(self.category_to_label[category])

        if not boxes:
            return None

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)
        target = {"boxes": boxes, "labels": labels}

        attrs = data["attributes"]
        global_attrs = {
            "weather": torch.tensor(self.weather_to_label.get(attrs["weather"], 0)),
            "scene": torch.tensor(self.scene_to_label.get(attrs["scene"], 0)),
            "timeofday": torch.tensor(self.timeofday_to_label.get(attrs["timeofday"], 0)),
        }

        if self.transform:
            image = self.transform(image)

        return image, target, global_attrs

###########################################
# 3. Classe MultiTaskModel (Detecção + Atributos Globais)
###########################################
class MultiTaskModel(nn.Module):
    def __init__(self, detection_model, backbone, num_weather, num_scene, num_time, in_features):
        super(MultiTaskModel, self).__init__()
        self.detection_model = detection_model
        self.backbone = backbone
        self.attr_pool = nn.AdaptiveAvgPool2d((1,1))
        self.fc_weather = nn.Linear(in_features, num_weather)
        self.fc_scene = nn.Linear(in_features, num_scene)
        self.fc_timeofday = nn.Linear(in_features, num_time)
    def forward(self, images, targets=None, global_attrs=None):
        # Em treinamento, retorna losses; em avaliação, retorna as detecções
        if self.training:
            detection_loss = self.detection_model(images, targets)
        else:
            detection_loss = self.detection_model(images)
        imgs_tensor = torch.stack(images)
        feats = self.backbone(imgs_tensor)
        pooled = self.attr_pool(feats)
        pooled = pooled.view(pooled.size(0), -1)
        weather_logits = self.fc_weather(pooled)
        scene_logits = self.fc_scene(pooled)
        timeofday_logits = self.fc_timeofday(pooled)
        if self.training and global_attrs is not None:
            weather_labels = torch.stack([attr["weather"] for attr in global_attrs]).to(images[0].device)
            scene_labels = torch.stack([attr["scene"] for attr in global_attrs]).to(images[0].device)
            timeofday_labels = torch.stack([attr["timeofday"] for attr in global_attrs]).to(images[0].device)
            loss_weather = nn.functional.cross_entropy(weather_logits, weather_labels)
            loss_scene = nn.functional.cross_entropy(scene_logits, scene_labels)
            loss_timeofday = nn.functional.cross_entropy(timeofday_logits, timeofday_labels)
            attr_loss = loss_weather + loss_scene + loss_timeofday
        else:
            attr_loss = 0
        return detection_loss, attr_loss, weather_logits, scene_logits, timeofday_logits

def collate_fn(batch):
    batch = [b for b in batch if b is not None]
    if not batch:
        return ([], [], [])
    images, targets, global_attrs = zip(*batch)
    return list(images), list(targets), list(global_attrs)

def compute_iou_torch(boxes1, boxes2):
    area1 = (boxes1[:, 2] - boxes1[:, 0]) * (boxes1[:, 3] - boxes1[:, 1])
    area2 = (boxes2[:, 2] - boxes2[:, 0]) * (boxes2[:, 3] - boxes2[:, 1])
    lt = torch.max(boxes1[:, None, :2], boxes2[:, :2])
    rb = torch.min(boxes1[:, None, 2:], boxes2[:, 2:])
    wh = (rb - lt).clamp(min=0)
    inter = wh[:, :, 0] * wh[:, :, 1]
    union = area1[:, None] + area2 - inter
    iou = inter / union
    return iou


import time

def evaluate(model, loader, device, model_type="default", iou_threshold=0.5, confidence_threshold=0.6):
    start_time = time.time()  # ⏱️ início

    model.eval()
    total = 0
    correct_weather, correct_scene, correct_time = 0, 0, 0
    total_gt = 0
    correct_category = 0
    detections_count = 0

    with torch.no_grad(), torch.cuda.amp.autocast():
        for batch_idx, (images, targets, attrs) in enumerate(loader):
            if not images:
                continue
            images = [img.to(device) for img in images]

            outputs = model(images)
            detections = outputs if isinstance(outputs, list) else model.detection_model(images)
            w_logits, s_logits, t_logits = model.fc_weather, model.fc_scene, model.fc_timeofday

            feats = model.backbone(torch.stack(images))
            pooled = model.attr_pool(feats).view(len(images), -1)
            w_preds = torch.argmax(w_logits(pooled), dim=1).to(device)
            s_preds = torch.argmax(s_logits(pooled), dim=1).to(device)
            t_preds = torch.argmax(t_logits(pooled), dim=1).to(device)

            w_true = torch.stack([a["weather"] for a in attrs]).to(device)
            s_true = torch.stack([a["scene"] for a in attrs]).to(device)
            t_true = torch.stack([a["timeofday"] for a in attrs]).to(device)

            correct_weather += (w_preds == w_true).sum().item()
            correct_scene += (s_preds == s_true).sum().item()
            correct_time += (t_preds == t_true).sum().item()
            total += len(images)

            for i in range(len(images)):
                gt_boxes = targets[i]["boxes"].to(device)
                gt_labels = targets[i]["labels"].to(device)
                total_gt += len(gt_boxes)

                pred_boxes = detections[i]["boxes"].to(device)
                pred_labels = detections[i]["labels"].to(device)
                pred_scores = detections[i]["scores"].to(device)

                keep = pred_scores >= confidence_threshold
                pred_boxes = pred_boxes[keep]
                pred_labels = pred_labels[keep]

                if len(gt_boxes) == 0 or len(pred_boxes) == 0:
                    continue

                ious = compute_iou_torch(gt_boxes, pred_boxes)
                max_iou, max_idx = ious.max(dim=1)
                matches = max_iou >= iou_threshold
                detections_count += matches.sum().item()
                correct_category += (pred_labels[max_idx[matches]] == gt_labels[matches]).sum().item()

            if batch_idx % 10 == 0:
                print(f"🔍 Avaliados {batch_idx+1}/{len(loader)} batches...")

    weather_acc = correct_weather / total if total > 0 else 0
    scene_acc = correct_scene / total if total > 0 else 0
    time_acc = correct_time / total if total > 0 else 0
    detection_cat_acc = correct_category / total_gt if total_gt > 0 else 0
    avg_detections = detections_count / total if total > 0 else 0

    total_time = time.time() - start_time  # ⏱️ fim
    print(f"\n⏱️ Tempo total de avaliação: {total_time:.2f} segundos")

    print(f"\n📊 Avaliação em {total} imagens:")
    print(f"🌤️ Acurácia Weather: {weather_acc:.2%}")
    print(f"🌆 Acurácia Scene:   {scene_acc:.2%}")
    print(f"🌙 Acurácia Time:    {time_acc:.2%}")
    print(f"📦 Média de detecções por imagem (matched): {avg_detections:.2f}")
    print(f"🎯 Acurácia de categorias: {detection_cat_acc:.2%}")

    os.makedirs("logs", exist_ok=True)
    evaluate_path = f"logs/evaluate_log_{model_type}.json"
    evaluation = {
        "images": total,
        "accuracy_weather": round(weather_acc, 4),
        "accuracy_scene": round(scene_acc, 4),
        "accuracy_time": round(time_acc, 4),
        "average_detections_per_image": round(avg_detections, 2),
        "accuracy_categories": round(detection_cat_acc, 4),
        "evaluation_time_seconds": round(total_time, 2)
    }
    with open(evaluate_path, "w") as f:
        json.dump(evaluation, f, indent=4)
    print(f"📄 Avaliação salva em: {evaluate_path}")

    return weather_acc, scene_acc, time_acc, avg_detections, detection_cat_acc

model_path = "multi_task_model_vgg_49.pth"
model_type = "vgg"
data_dir = "images/val"
label_dir = "labels/val"
batch_size = 8

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"📦 Carregando modelo salvo de: {model_path}")
model = torch.load(model_path, map_location=device, weights_only=False)
model = model.to(device)

transform = transforms.Compose([transforms.ToTensor()])
dataset = MultiTaskObjectDetectionDataset(data_dir, label_dir, transform)
subset_size = int(len(dataset) * 0.5)
indices = np.random.choice(len(dataset), subset_size, replace=False)
subset = Subset(dataset, indices)
loader = DataLoader(subset, batch_size=batch_size, collate_fn=collate_fn)

print("🚀 Iniciando avaliação...")
evaluate(model, loader, device, model_type)

📦 Carregando modelo salvo de: multi_task_model_vgg_49.pth
Todas as imagens possuem JSON correspondente.
🚀 Iniciando avaliação...


C:\Users\ctw02813\AppData\Local\Temp\ipykernel_36860\3222666317.py:168: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast():
C:\Users\ctw02813\AppData\Roaming\Python\Python312\site-packages\torch\amp\autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(


🔍 Avaliados 1/63 batches...
🔍 Avaliados 11/63 batches...
🔍 Avaliados 21/63 batches...
🔍 Avaliados 31/63 batches...
🔍 Avaliados 41/63 batches...
🔍 Avaliados 51/63 batches...
🔍 Avaliados 61/63 batches...

⏱️ Tempo total de avaliação: 3792.21 segundos

📊 Avaliação em 498 imagens:
🌤️ Acurácia Weather: 56.02%
🌆 Acurácia Scene:   65.06%
🌙 Acurácia Time:    89.56%
📦 Média de detecções por imagem (matched): 3.79
🎯 Acurácia de categorias: 23.57%
📄 Avaliação salva em: logs/evaluate_log_vgg.json


(0.5602409638554217,
 0.6506024096385542,
 0.8955823293172691,
 3.789156626506024,
 0.23570247933884297)